In [ ]:
from src.common.constants import Constants as consts
from src.preprocessing.experiment_preprocessor import ExperimentPreprocessor

preprocessing_config = {
    "splits_names": ["train", "dev"],
    "fraction": 0.05,
    "is_audio_ids_sampling": False,
    "balance_splits_config": None,
}

exp_preprocessor = ExperimentPreprocessor(
    load_file_name=consts.feature_extracted,
    save_file_name=consts.feature_extracted,
    feat_suffix=consts.wavlm_emb_suffix
)

prep_data = exp_preprocessor.preprocess_data(**preprocessing_config)
print(prep_data.keys())

In [3]:
meta_train, feat_train = prep_data["train"]
print(feat_train.shape, meta_train.shape)

(38599, 768) (38599, 10)


In [8]:
balance_strategy_ratios = {
    "unbalanced": [None],
    "oversample": [0.5, 0.75, 1.0],
    "undersample": [0.5, 0.75, 1.0],
    "mix": [[0.5, 0.75], [0.5, 1.0], [0.75, 1.0]],
}

balance_strategy_ratios = {
    "undersample": [0.5, 0.75, 1.0],
}

In [7]:
from src.models.model_trainer import ModelTrainer
from src.models.logistic_regression_clf import LogisticRegressionClf, objective_acc
from src.common.constants import ExperimentPreprocessConfig

trainer = ModelTrainer()

def create_comparision_dict(balance_strategy_ratios):
    comparison_dict = {}
    for strategy, ratios in balance_strategy_ratios.items():
        comparison_dict[f"{strategy}:{ratios}"] = ratios
    return comparison_dict

def train_all_balancers(
    balance_strategy_ratios,
    exp_preprocessor: ExperimentPreprocessor,
    exp_name="best_balance_pipeline",
):
    comparison_dict = create_comparision_dict(balance_strategy_ratios)

    for strategy, ratios in balance_strategy_ratios.items():
        for ratio in ratios:
            preprocessing_config = ExperimentPreprocessConfig(
                splits_names=["train", "dev"],
                fraction=0.05,
                is_audio_ids_sampling=False,
                balance_splits_config=[strategy, ratio],
            )

            data_for_exp = exp_preprocessor.preprocess_data(**preprocessing_config.get_dict())
            meta_train, X_train = data_for_exp["train"]
            y_train = trainer.get_target(metadata=meta_train)
            meta_dev, X_dev = data_for_exp["dev"]
            y_dev = trainer.get_target(metadata=meta_dev)

            trainer.optuna_train(
                model=LogisticRegressionClf,
                objective=objective_acc,
                n_trials=20,
                max_iter=100,
                X_train=X_train,
                y_train=y_train,
                X_dev=X_dev,
                y_dev=y_dev,
                )

            best_value = trainer.get_best_value()
            comparison_dict_key = f"{strategy}:{ratio}"
            comparison_dict[comparison_dict_key] = best_value

    return comparison_dict

results = train_all_balancers(balance_strategy_ratios, exp_preprocessor)
print(results)

2026-03-19 20:58:40 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_train.csv
INFO:FeatureLoader:Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_train.csv
2026-03-19 20:58:40 | INFO     | FeatureLoader | Loading features from /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_train.csv
INFO:FeatureLoader:Loading features from /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_train.csv
2026-03-19 20:58:40 | INFO     | FeatureLoader | Loading metadata from /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_train.csv
INFO:FeatureLoader:Loading metadata from /Users/mikolajkarapka/Projects/audio-deepfake-detection-u

Exception: Unknown balancer type: BalanceType.UNDERSAMPLE